# Importations alimentaires des Comores

Notebook du projet FAOSTAT. Les données brutes sont conservées sans modification.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent
RAW_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
FIGURES_DIR = ROOT_DIR / "figures"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Chargement

In [2]:
FILE = PROCESSED_DIR / "imports_comoros_processed.csv"
if not FILE.exists(): raise FileNotFoundError("Exécute le notebook 01.")
data = pd.read_csv(FILE)
supplier = data.dropna(subset=["Year","Item Code (CPC)","Partner Countries","Value (1000 USD)"]).copy()
supplier = supplier[supplier["Value (1000 USD)"] >= 0]

## 2. Calcul de CR1, CR3 et HHI

In [3]:
def concentration_metrics(group):
    amounts = group.groupby("Partner Countries")["Value (1000 USD)"].sum().sort_values(ascending=False)
    total = amounts.sum()
    if total <= 0:
        return pd.Series({"total_value_1000_usd": total, "CR1": np.nan,
                          "CR3": np.nan, "HHI": np.nan, "suppliers": len(amounts)})
    shares = amounts / total
    return pd.Series({"total_value_1000_usd": total,
                      "CR1": shares.head(1).sum(),
                      "CR3": shares.head(3).sum(),
                      "HHI": (shares**2).sum(),
                      "suppliers": len(shares)})

concentration = supplier.groupby(["Year","Item Code (CPC)","Item"]).apply(
    concentration_metrics
).reset_index()
display(concentration.head())

C:\Users\Nael\AppData\Local\Temp\ipykernel_5296\1895539595.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  concentration = supplier.groupby(["Year","Item Code (CPC)","Item"]).apply(


,Year,Item Code (CPC),Item,total_value_1000_usd,CR1,CR3,HHI,suppliers
0,1995,0111,Wheat,0.00,NaN,NaN,NaN,1.00
1,1995,0112,Maize (corn),13.00,0.69,1.00,0.53,5.00
2,1995,0113,Rice,78.00,0.33,0.90,0.28,5.00
3,1995,01199.90,Cereals n.e.c.,0.00,NaN,NaN,NaN,1.00
4,1995,01213,Cauliflowers and broccoli,0.00,NaN,NaN,NaN,3.00


## 3. Produits les plus concentrés lors de la dernière année

In [4]:
last_year = supplier["Year"].max()
latest = concentration[concentration["Year"] == last_year].copy()
display(latest.sort_values("CR1", ascending=False).head(20))

,Year,Item Code (CPC),Item,total_value_1000_usd,CR1,CR3,HHI,suppliers
2313,2021,0112,Maize (corn),3.00,1.00,1.00,1.00,2.00
2318,2021,01231,"Chillies and peppers, green (Capsicum spp. and...",1.00,1.00,1.00,1.00,1.00
2321,2021,01241.90,"Other beans, green",5.00,1.00,1.00,1.00,2.00
2320,2021,01234,Tomatoes,1.00,1.00,1.00,1.00,2.00
2332,2021,01330,Grapes,1.00,1.00,1.00,1.00,1.00
2325,2021,01254,Leeks and other alliaceous vegetables,1.00,1.00,1.00,1.00,2.00
2330,2021,01323,Oranges,1.00,1.00,1.00,1.00,1.00
2329,2021,01322,Lemons and limes,2.00,1.00,1.00,1.00,2.00
2349,2021,01599.10,Edible roots and tubers with high starch or in...,17.00,1.00,1.00,1.00,1.00
2353,2021,01651,"Pepper (Piper spp.), raw",4.00,1.00,1.00,1.00,4.00


## 4. Examiner un produit

In [5]:
selected_cpc = None  # Exemple : "0113"
if selected_cpc is not None:
    subset = supplier[supplier["Item Code (CPC)"].astype(str) == str(selected_cpc)]
    shares = subset.groupby("Partner Countries")["Value (1000 USD)"].sum()
    shares = (shares / shares.sum()).sort_values(ascending=False)
    display(shares.to_frame("Part"))
    shares.head(10).sort_values().plot(kind="barh", figsize=(9,5))
    plt.title(f"Fournisseurs du produit CPC {selected_cpc}")
    plt.tight_layout(); plt.show()

In [6]:
concentration.to_csv(PROCESSED_DIR / "supplier_concentration.csv", index=False, encoding="utf-8-sig")
print("Indicateurs de concentration exportés.")

Indicateurs de concentration exportés.
